In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Usando o caminho completo do seu computador
df = pd.read_excel("base_vendas_supermercado.xlsx")

print(f"Base carregada: {df.shape[0]} linhas | {df.shape[1]} colunas")
print(f"Colunas: {list(df.columns)}")


np.random.seed(42)
df_raw = df.copy()

# Inserindo NaN em colunas específicas
for col, n in [('Categoria', 10), ('Quantidade', 8)]:
    idx = np.random.choice(df_raw.index, n, replace=False)
    df_raw.loc[idx, col] = np.nan

# Inserindo valores absurdos (outliers)
idx_out = np.random.choice(df_raw.index, 3, replace=False)
df_raw.loc[idx_out, 'Valor Líquido'] = [5000, 4500, 0.01]

# Inserindo duplicatas
dup = df_raw.sample(5, random_state=1)
df_raw = pd.concat([df_raw, dup], ignore_index=True)

print(f"Base suja criada:")
print(f"  Linhas     : {df_raw.shape[0]}")
print(f"  NaN total  : {df_raw.isnull().sum().sum()}")
print(f"  Duplicatas : {df_raw.duplicated().sum()}")
print(f"  Máx Valor  : R$ {df_raw['Valor Líquido'].max():.2f}")
print(f"  Mín Valor  : R$ {df_raw['Valor Líquido'].min():.2f}")


# --- NaN por coluna ---
nan_contagem = df_raw.isnull().sum()
nan_pct      = (df_raw.isnull().mean() * 100).round(1)

print("=== Percentual de NaN por coluna ===")
for col in df_raw.columns:
    if nan_contagem[col] > 0:
        print(f"  {col:<25} {nan_contagem[col]:>3} NaN  ({nan_pct[col]:>4.1f}%)")

print()

# --- Duplicatas ---
n_dup = df_raw.duplicated().sum()
print(f"=== Duplicatas ===")
print(f"  Total de linhas duplicadas: {n_dup}")


df_clean = df_raw.copy()

# Etapa 1: remover duplicatas
antes = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Duplicatas removidas: {antes - len(df_clean)} linhas")

# Etapa 2: preencher NaN de texto com a moda
moda_categoria = df_clean['Categoria'].mode()[0]
df_clean['Categoria'] = df_clean['Categoria'].fillna(moda_categoria)
print(f"Categoria: NaN preenchidos com '{moda_categoria}'")

# Etapa 3: preencher NaN numérico com a mediana
mediana_qtd = df_clean['Quantidade'].median()
df_clean['Quantidade'] = df_clean['Quantidade'].fillna(mediana_qtd)
print(f"Quantidade: NaN preenchidos com {mediana_qtd}")

# Etapa 4: filtrar outliers de Valor Líquido
antes = len(df_clean)
df_clean = df_clean[(df_clean['Valor Líquido'] >= 1) & (df_clean['Valor Líquido'] <= 300)].copy()
print(f"Outliers removidos: {antes - len(df_clean)} linhas")

print()
print("=== Resumo após limpeza ===")
print(f"  Linhas   : {len(df_clean)}")
print(f"  NaN total: {df_clean.isnull().sum().sum()}")
print(f"  Dup      : {df_clean.duplicated().sum()}")
print(f"  Valor mín: R$ {df_clean['Valor Líquido'].min():.2f}")
print(f"  Valor máx: R$ {df_clean['Valor Líquido'].max():.2f}")


# Coluna 1: número do mês
df_clean['Mês_Num'] = df_clean['Data'].dt.month

# Coluna 2: perfil de compra por faixa de valor
df_clean['Perfil_Compra'] = pd.cut(
    df_clean['Valor Líquido'],
    bins=[0, 25, 80, 300],
    labels=['Pequeno', 'Médio', 'Grande']
)

# Verificando o resultado
print("Amostra das novas colunas:")
print(df_clean[['Data', 'Mês_Num', 'Valor Líquido', 'Perfil_Compra']].head(8).to_string(index=False))

print()
print("Distribuição por Perfil de Compra:")
print(df_clean['Perfil_Compra'].value_counts().sort_index().to_string())


# Agrupando e somando por categoria
vendas_cat = (
    df_clean
    .groupby('Categoria')['Valor Líquido']
    .sum()
    .sort_values(ascending=False)
)

print("Total de vendas por categoria:")
for cat, val in vendas_cat.items():
    print(f"  {cat:<20} R$ {val:>8.2f}")

print()

# --- Gráfico ---
fig, ax = plt.subplots(figsize=(9, 5))

cores = ['#3b82f6', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6', '#ec4899', '#06b6d4']

bars = ax.barh(
    vendas_cat.index[::-1],    # invertemos para o maior ficar no topo
    vendas_cat.values[::-1],
    color=cores[:len(vendas_cat)],
    edgecolor='white',
    height=0.6
)

# Anotações de valor em cada barra
max_val = vendas_cat.max()
for i, val in enumerate(vendas_cat.values[::-1]):
    ax.text(val + max_val * 0.01, i, f'R$ {val:.0f}', va='center', fontsize=9, fontweight='bold')

ax.set_title('Total de Vendas por Categoria', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Valor Líquido Total (R$)')
ax.set_xlim(0, max_val * 1.18)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()



FileNotFoundError: [Errno 2] No such file or directory: 'base_vendas_supermercado.xlsx'